# Round 1 Trading Groundwork

This notebook documents the first Intara algorithm for `ASH_COATED_OSMIUM` and `INTARIAN_PEPPER_ROOT`. The goal is an explainable strategy that uses the round data directly, avoids overfit ML, and can be translated into the competition `Trader` class.

Core idea: use product-specific structure. Pepper root has a persistent slow growth drift; osmium is stationary around 10,000 with short-horizon mean reversion and order-book imbalance effects.

## Final Algorithmic Strategy

Upload `trader.py` for the algorithmic round. The bot trades only `ASH_COATED_OSMIUM` and `INTARIAN_PEPPER_ROOT`, respects the 80-unit limits, and stores only compact state in `traderData`.

- `INTARIAN_PEPPER_ROOT`: estimate fair value as `intercept + 0.001 * timestamp`, where the intercept is recalibrated from the live book. Build to the long limit while the book is not too expensive, exit patiently after timestamp `995000`, then use a wider forced exit after `998000` to avoid relying on terminal mark-to-market inventory.
- `ASH_COATED_OSMIUM`: treat as stationary around 10,000. Use an anchored EMA and a small order-book imbalance adjustment, then only cross when the edge is wide enough.
- Risk posture: the 200k objective is carried by the pepper-root drift; osmium is deliberately secondary and should not be required for target achievement.


## Theorem-Style Findings

**Finding 1: Pepper root has a stable affine fair-value law.** Across days `-2`, `-1`, and `0`, same-timestamp pepper-root mids have correlation about `0.9999`, and the day-to-day level shift is about `1,000`. Within each day, the visible slope is approximately `1 XIREC per 1,000 timestamp units`. This supports the fair-value form `intercept + 0.001 * timestamp`.

**Train/test protocol.** Days `-2` and `-1` are used for parameter selection. Day `0` is treated as the local holdout test run, because the live submission is expected to be evaluated on day `1`.

**Trading theorem: if the live day preserves this affine drift and the bot can buy near the fair path early, the best bounded-risk posture is to hold the maximum allowed long inventory for most of the day, then flatten late.** With an 80-unit position limit and roughly 1,000 XIREC intraday drift, the gross opportunity is near `80,000` XIRECs per day before spread and exit costs. The staged-exit backtest realizes about `78.8k` per day while ending flat.

**Finding 2: Osmium is stationary and mean-reverting, not a primary directional edge.** Nonzero mids cluster tightly around 10,000; EMA-deviation versus next-tick delta is negatively correlated, and order-book imbalance is positively correlated with next-tick delta. That supports conservative mean reversion and passive quoting, but the diagnostics intentionally do not require osmium profit to clear the round target.

**Overfit guardrail.** The selected parameters rank `1 / 48` by the train-only robust score on days `-2` and `-1`, but only `7 / 48` by all-days combined PnL. This is the intended behavior: the chosen values are on a stable plateau, not selected because they maximize day `0`. The selected neighborhood has train median `157,594` and train minimum `157,353`, while holdout day `0` remains `78,774`.

**Robustness theorem.** Under the implemented crossing-fill replay, if fills are reduced to 97%, adverse slippage is up to 1 XIREC, and terminal marks receive 3 XIREC Gaussian noise, the Monte Carlo 5th percentile remains above `236k`. Therefore, the strategy has a wide margin to the 200k target under these execution assumptions.


In [1]:
import csv, math, statistics, sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / 'trader.py').exists() and (ROOT.parent / 'trader.py').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
DATA = ROOT / 'data' / 'round1'

price_rows = []
for path in sorted(DATA.glob('prices_round_1_day_*.csv')):
    with path.open(newline='') as f:
        for row in csv.DictReader(f, delimiter=';'):
            parsed = {
                'day': int(row['day']),
                'timestamp': int(row['timestamp']),
                'product': row['product'],
                'mid_price': float(row['mid_price']),
            }
            for level in (1, 2, 3):
                for side in ('bid', 'ask'):
                    parsed[f'{side}_price_{level}'] = float(row[f'{side}_price_{level}']) if row[f'{side}_price_{level}'] else None
                    parsed[f'{side}_volume_{level}'] = float(row[f'{side}_volume_{level}']) if row[f'{side}_volume_{level}'] else None
            price_rows.append(parsed)

products = sorted({row['product'] for row in price_rows})
print(f'Loaded {len(price_rows)} price rows from 3 files')
print('Products:', ', '.join(products))


Loaded 60000 price rows from 3 files\nProducts: ASH_COATED_OSMIUM, INTARIAN_PEPPER_ROOT\n

In [2]:
for product in products:
    print(product)
    for day in sorted({row['day'] for row in price_rows}):
        mids = [row['mid_price'] for row in price_rows if row['product'] == product and row['day'] == day and row['mid_price'] > 0]
        zeros = sum(1 for row in price_rows if row['product'] == product and row['day'] == day and row['mid_price'] <= 0)
        print(
            f' day {day:2d}: n={len(mids)} zeros={zeros} mean={statistics.mean(mids):.2f} '
            f'std={statistics.pstdev(mids):.2f} min={min(mids):.1f} max={max(mids):.1f} '
            f'first={mids[0]:.1f} last={mids[-1]:.1f}'
        )
    print()

ASH_COATED_OSMIUM\n day -2: n=9982 zeros=18 mean=9998.17 std=5.22 min=9979.0 max=10019.0 first=10010.0 last=9993.5\n day -1: n=9983 zeros=17 mean=10000.83 std=4.45 min=9982.0 max=10019.0 first=10003.0 last=10002.0\n day  0: n=9986 zeros=14 mean=10001.61 std=5.68 min=9977.0 max=10023.0 first=10013.0 last=10007.0\n\nINTARIAN_PEPPER_ROOT\n day -2: n=9984 zeros=16 mean=10499.96 std=288.71 min=9998.5 max=11003.0 first=9998.5 last=11001.5\n day -1: n=9983 zeros=17 mean=11500.03 std=288.65 min=10995.0 max=12006.0 first=10998.5 last=11998.0\n day  0: n=9979 zeros=21 mean=12500.17 std=288.72 min=11994.0 max=13007.0 first=11998.5 last=13000.0\n

## Product Diagnostics

`INTARIAN_PEPPER_ROOT` is not flat in the historical data; it is steady in the sense that it follows a remarkably stable upward line. Each day starts about 1,000 XIRECs above the prior day and rises about another 1,000 XIRECs intraday, which implies a slope near `0.001 * timestamp`.

`ASH_COATED_OSMIUM` is centered near 10,000 with a much tighter nonzero mid-price range. Its edge should come from patient mean reversion and spread capture, not a directional forecast.

In [3]:
def corr(xs, ys):
    mx, my = statistics.mean(xs), statistics.mean(ys)
    sx = sum((x - mx) ** 2 for x in xs)
    sy = sum((y - my) ** 2 for y in ys)
    return sum((x - mx) * (y - my) for x, y in zip(xs, ys)) / (sx * sy) ** 0.5 if sx and sy else 0.0

for product in products:
    print(product)
    by_day = {}
    for day in (-2, -1, 0):
        by_day[day] = {row['timestamp']: row['mid_price'] for row in price_rows if row['product'] == product and row['day'] == day and row['mid_price'] > 0}
    for left, right in [(-2, -1), (-1, 0)]:
        stamps = sorted(set(by_day[left]).intersection(by_day[right]))
        xs = [by_day[left][stamp] for stamp in stamps]
        ys = [by_day[right][stamp] for stamp in stamps]
        print(f' day-pair {left}/{right}: same-timestamp corr={corr(xs, ys):.4f} mean_diff={statistics.mean(y - x for x, y in zip(xs, ys)):.2f}')

    ema_dev, imbalance, next_delta = [], [], []
    for day in (-2, -1, 0):
        rows = [row for row in price_rows if row['product'] == product and row['day'] == day and row['mid_price'] > 0]
        rows.sort(key=lambda row: row['timestamp'])
        ema = rows[0]['mid_price']
        for i, row in enumerate(rows[:-1]):
            bid, ask = row['bid_price_1'], row['ask_price_1']
            bid_volume, ask_volume = row['bid_volume_1'], row['ask_volume_1']
            if bid is None or ask is None:
                continue
            ema_dev.append(row['mid_price'] - ema)
            imbalance.append((bid_volume - ask_volume) / (bid_volume + ask_volume))
            next_delta.append(rows[i + 1]['mid_price'] - row['mid_price'])
            ema = 0.8 * ema + 0.2 * row['mid_price']
    print(f' signal correlations: ema_deviation_vs_next_delta={corr(ema_dev, next_delta):.4f} imbalance_vs_next_delta={corr(imbalance, next_delta):.4f}')
    print()

ASH_COATED_OSMIUM\n day-pair -2/-1: same-timestamp corr=-0.0598 mean_diff=2.66\n day-pair -1/0: same-timestamp corr=-0.0644 mean_diff=0.78\n signal correlations: ema_deviation_vs_next_delta=-0.4077 imbalance_vs_next_delta=0.3809\n\nINTARIAN_PEPPER_ROOT\n day-pair -2/-1: same-timestamp corr=0.9999 mean_diff=1000.01\n day-pair -1/0: same-timestamp corr=0.9999 mean_diff=999.99\n signal correlations: ema_deviation_vs_next_delta=-0.4530 imbalance_vs_next_delta=0.3849\n

## Strategy Translation

Pepper root fair value is modeled as `intercept + 0.001 * timestamp`, where the intercept is recalibrated from the live book. The bot builds to the 80-unit long limit when asks are no more than 8 XIRECs above the current fair value, then exits near the end of the day. This captures the steady growth without needing a fragile regression model.

Osmium fair value is an anchored EMA around 10,000 with a small order-book imbalance adjustment. The bot only crosses when the edge is at least 4 XIRECs, otherwise it posts inventory-skewed passive quotes and flattens near the close.

In [4]:
import json
from pathlib import Path

diagnostics = json.loads(Path('round1_diagnostics.json').read_text()) if Path('round1_diagnostics.json').exists() else json.loads(Path('logs/round1_diagnostics.json').read_text())
mc = diagnostics['monte_carlo']
grid = diagnostics['parameter_grid']
selected = grid['selected']
print('selection protocol:', grid['selection_protocol'])
print('deterministic combined pnl:', diagnostics['deterministic_backtest']['combined_pnl'])
print('train target pass rate:', grid['train_target_pass_rate'])
print('all-days target pass rate:', grid['target_pass_rate'])
print('selected train rank:', grid['selected_train_rank'], 'of', grid['summary']['count'])
print('selected all-days combined rank:', grid['selected_combined_rank'], 'of', grid['summary']['count'])
print('selected train total:', selected['train_total_pnl'])
print('selected holdout day 0:', selected['holdout_day0_pnl'])
print('selected neighborhood train summary:', grid['selected_neighborhood_train_summary'])
print('mc summary:', mc['summary'])
print('mc probability >= 200k:', mc['probability_above_200k'])
print('Gelman-Rubin R-hat:', mc['gelman_rubin_rhat'])
print('Geweke:', mc['geweke'])
print('Anderson-Darling:', mc['anderson_darling_normality'])
print('K-S:', mc['kolmogorov_smirnov_normality'])



selection protocol: Select parameters using only days -2 and -1. Day 0 is treated as a holdout test run, because live submission is expected to be evaluated on day 1.
deterministic combined pnl: 236444.0
train target pass rate: 1.0
all-days target pass rate: 1.0
selected train rank: 1 of 48
selected all-days combined rank: 7 of 48
selected train total: 157670.0
selected holdout day 0: 78774.0
selected neighborhood train summary: {'count': 18, 'min': 157353.0, 'p05': 157353.0, 'p10': 157447.5, 'median': 157594.0, 'mean': 157599.66666666666, 'p90': 157776.0, 'p95': 157776.0, 'max': 157776.0, 'stdev': 130.72022711798576}
mc summary: {'count': 160, 'min': 236125.0, 'p05': 236152.94999999998, 'p10': 236163.0, 'median': 236209.0, 'mean': 236207.25625, 'p90': 236246.0, 'p95': 236256.2, 'max': 236348.0, 'stdev': 34.54476206225048}
mc probability >= 200k: 1.0
Gelman-Rubin R-hat: 1.0
Geweke: [{'first_count': 4, 'last_count': 20, 'z_score': -0.6283004503020156, 'passes_abs_z_lt_2': True}, {'first

## Why This Should Work

The main profit source is structural rather than overfit: buying pepper root early and selling late captures a repeated intraday drift that is visible on every training day. The model estimates the level from the current book, so it does not depend on knowing the hidden live day number.

Osmium is deliberately secondary. Its stationary distribution and negative EMA-deviation correlation support mean reversion, while the wider crossing threshold avoids taking noisy trades that the replay could not justify. Passive quotes may add upside, but the plan does not rely on them for the 200,000 XIREC target.

## Robustness Diagnostics

The committed diagnostics script runs the strategy through deterministic replay, a 48-point parameter sensitivity grid, Monte Carlo execution stress, Geweke stability checks, Gelman-Rubin R-hat, Anderson-Darling normality testing, and a K-S normality cross-check. Geweke and R-hat are used here to check Monte Carlo simulation stability, not as magical proof of market stationarity.

Parameter selection uses only days `-2` and `-1`. Day `0` is reported as the holdout test run. The latest 4-chain x 40-draw Monte Carlo run applies 97% crossing-fill probability, up to 1 XIREC adverse slippage, and 3 XIREC mark noise. The 5th percentile remains above 236k, so the 200k target is not coming from a fragile single replay.


In [5]:
import json
from pathlib import Path

diagnostics = json.loads(Path('round1_diagnostics.json').read_text()) if Path('round1_diagnostics.json').exists() else json.loads(Path('logs/round1_diagnostics.json').read_text())
mc = diagnostics['monte_carlo']
grid = diagnostics['parameter_grid']
selected = grid['selected']
print('selection protocol:', grid['selection_protocol'])
print('deterministic combined pnl:', diagnostics['deterministic_backtest']['combined_pnl'])
print('train target pass rate:', grid['train_target_pass_rate'])
print('all-days target pass rate:', grid['target_pass_rate'])
print('selected train rank:', grid['selected_train_rank'], 'of', grid['summary']['count'])
print('selected all-days combined rank:', grid['selected_combined_rank'], 'of', grid['summary']['count'])
print('selected train total:', selected['train_total_pnl'])
print('selected holdout day 0:', selected['holdout_day0_pnl'])
print('selected neighborhood train summary:', grid['selected_neighborhood_train_summary'])
print('mc summary:', mc['summary'])
print('mc probability >= 200k:', mc['probability_above_200k'])
print('Gelman-Rubin R-hat:', mc['gelman_rubin_rhat'])
print('Geweke:', mc['geweke'])
print('Anderson-Darling:', mc['anderson_darling_normality'])
print('K-S:', mc['kolmogorov_smirnov_normality'])



selection protocol: Select parameters using only days -2 and -1. Day 0 is treated as a holdout test run, because live submission is expected to be evaluated on day 1.
deterministic combined pnl: 236444.0
train target pass rate: 1.0
all-days target pass rate: 1.0
selected train rank: 1 of 48
selected all-days combined rank: 7 of 48
selected train total: 157670.0
selected holdout day 0: 78774.0
selected neighborhood train summary: {'count': 18, 'min': 157353.0, 'p05': 157353.0, 'p10': 157447.5, 'median': 157594.0, 'mean': 157599.66666666666, 'p90': 157776.0, 'p95': 157776.0, 'max': 157776.0, 'stdev': 130.72022711798576}
mc summary: {'count': 160, 'min': 236125.0, 'p05': 236152.94999999998, 'p10': 236163.0, 'median': 236209.0, 'mean': 236207.25625, 'p90': 236246.0, 'p95': 236256.2, 'max': 236348.0, 'stdev': 34.54476206225048}
mc probability >= 200k: 1.0
Gelman-Rubin R-hat: 1.0
Geweke: [{'first_count': 4, 'last_count': 20, 'z_score': -0.6283004503020156, 'passes_abs_z_lt_2': True}, {'first